# K1a — 직접 모순 합성 학습 데이터

## 왜 하나

M1a에서 최신 LLM(Claude Haiku·Sonnet 5.5) 환각을 판정기가 놓친 원인을 보니, **놓친 진짜 환각의 68%가 직접 모순**이었다.
근거에 있는 단어를 그대로 쓰면서 *누가·무엇을*만 바꾼 문장이다(같은 리뷰어의 말을 "다른 리뷰어도"로, 인물 역할 뒤바꿈). 모순 46문장 중 경보는 1개.
학습 데이터(RAGTruth, 2023년 무렵 LLM 응답)에 이런 미세한 모순이 적었을 가능성이 크다 → **이런 모순을 합성해 F1 base를 한 번 더 학습**한다(K1b).

## 무엇을 하나

1. gold-large 학습 응답(train 분할, cal-L·dev-L 근거 제외)에서 과제마다 500개, 응답당 멀쩡한 문장 최대 2개를 고른다
2. `claude-sonnet-5-5`(배치)가 문장마다 **① 최소 수정 모순**(바뀐 구절 표시)과 **② 충실한 바꿔쓰기**(대조용)를 만든다
3. 그 문장만 바꾼 응답을 R4와 같은 정답 지시문으로 `claude-sonnet-5-5`가 **따로** 판정 → ①은 H, ②는 S인 것만 남긴다
4. 통과한 ①·②를 같은 수(최대 각 1,500)로 `k1/synth_train.json`에 저장, 가린 점검용 모순 40개를 `k1_check.json`으로 내보낸다

## 사전 정의 기준 (SSOT §6 K1, 실행 전 고정)

- **K1-0(이 노트북 부분)**: 검증을 통과한 모순 ≥ 300개. 그리고 `k1_check.json` 40개를 하위 에이전트가 가린 채 점검해 진짜 모순 ≥ 85% (점검은 Claude Code에서)
- 주 기준 K1-1(새 생성 환각 응답 탐지율 +0.10)과 안전 기준 K1-2·K1-3은 K1b 노트북에서 판정

## 실행 방법

1. 왼쪽 열쇠 아이콘(보안 비밀)에 `ANTHROPIC_API_KEY`, 노트북 접근 켜기
2. 런타임: **CPU로 충분**(GPU 필요 없음)
3. **모두 실행**. 배치 두 번(생성 → 검증), 보통 1~2시간. 끊기면 다시 모두 실행 — 드라이브 `k1/`에서 이어서 한다
4. 비용(추정, 실행 중 출력): 생성 약 $7 + 검증 약 $15
5. 끝나면 `k1a_results.zip`이 내려받아진다 → Claude Code에 올리기 (`k1_check.json` 점검)

> 드라이브에 `r0/r0_items.pkl`, `r4b/labels_train.json`이 있어야 한다. 이 노트북과 별도로 `m1_fresh_eval.ipynb`(새 평가 세트)를 돌려 둔다.


## 1. 환경 준비

In [ ]:
!pip -q install anthropic pysbd tiktoken
!git clone --depth 1 https://github.com/ParticleMedia/RAGTruth.git 2>/dev/null || echo 'RAGTruth already cloned'
import json, time, random, os, gc, copy, pickle
import numpy as np, pandas as pd, pysbd, tiktoken
from google.colab import drive
drive.mount('/content/drive')
SEED = 42
random.seed(SEED); np.random.seed(SEED)
OUT_DIR = '/content/drive/MyDrive/Colab Notebooks/HalluGate'
R0_DIR, R4B_DIR, K1_DIR = (os.path.join(OUT_DIR, d) for d in ['r0', 'r4b', 'k1'])
os.makedirs(K1_DIR, exist_ok=True)
for p in [os.path.join(R4B_DIR, 'labels_train.json'), os.path.join(R0_DIR, 'r0_items.pkl')]:
    assert os.path.exists(p), f'{p} 없음'
def save_json(o, p): json.dump(o, open(p + '.tmp', 'w'), ensure_ascii=False); os.replace(p + '.tmp', p)
def load_json(p): return json.load(open(p)) if os.path.exists(p) else None
R0P = pickle.load(open(os.path.join(R0_DIR, 'r0_items.pkl'), 'rb'))

## 2. 데이터와 보정 학습 정답 (R4c·F1과 같음)

In [ ]:
BASE = 'RAGTruth/dataset'
responses = [json.loads(l) for l in open(f'{BASE}/response.jsonl')]
sources   = {json.loads(l)['source_id']: json.loads(l) for l in open(f'{BASE}/source_info.jsonl')}
for r in responses:
    r['task_type'] = sources[r['source_id']]['task_type']

def build_reference(src):
    si, tt = src['source_info'], src['task_type']
    if tt == 'QA':
        return f"Question: {si.get('question','')}\n{si.get('passages','')}"
    if tt == 'Summary':
        return si if isinstance(si, str) else json.dumps(si, ensure_ascii=False)
    return '\n'.join(f"{k}: {json.dumps(v, ensure_ascii=False) if isinstance(v,(dict,list)) else v}"
                     for k, v in (si.items() if isinstance(si, dict) else []))

TASKS = ['QA', 'Summary', 'Data2txt']
train_pool = [r for r in responses if r['split'] == 'train']          # test 분할은 사용하지 않음
random.seed(SEED)
dev = []
for tt in TASKS:
    dev += random.sample([r for r in train_pool if r['task_type'] == tt], 120)
dev_ids = {r['id'] for r in dev}
rest = [r for r in train_pool if r['id'] not in dev_ids]
rng = random.Random(11)
teacher_resp = []
for tt in TASKS:
    teacher_resp += rng.sample([r for r in rest if r['task_type'] == tt], 200)
rng.shuffle(teacher_resp)
val = teacher_resp[:int(len(teacher_resp) * 0.15)]                   # student 학습에 쓰지 않은 검증 분할
assert not (dev_ids & {r['id'] for r in val})
print(f'dev {len(dev)} 응답 | 검증 {len(val)} 응답')

In [ ]:
# 학습 응답: S2/S2-b의 gold-large와 같은 구성 (teacher 학습 510 + 추가 2,490). dev·검증과 겹치지 않음을 확인한다.
teacher_train = teacher_resp[len(val):]
used = dev_ids | {r['id'] for r in teacher_resp}
gl_extra = rng.sample([r for r in rest if r['id'] not in used], 3000 - len(teacher_train))
train = teacher_train + gl_extra
tr_ids = {r['id'] for r in train}
assert not (tr_ids & dev_ids) and not (tr_ids & {r['id'] for r in val}), '학습과 dev/검증 겹침'
print(f'학습 {len(train)} 응답 (dev·검증과 겹침 없음)')

In [ ]:
ENC = tiktoken.get_encoding('cl100k_base')
seg = pysbd.Segmenter(language='en', clean=False, char_span=True)

def tokenize_offsets(text):
    ids = ENC.encode(text)
    dec, starts = ENC.decode_with_offsets(ids)
    if dec != text:                       # 드문 경우: 멀티바이트 문자 경계 — 누적 디코드로 대체
        starts, pos = [], 0
        for i in range(len(ids)):
            starts.append(pos); pos = len(ENC.decode(ids[:i + 1]))
    ends = starts[1:] + [len(text)]
    return starts, ends

def prep(r):
    text = r['response']
    ts, te = tokenize_offsets(text)
    # 토큰의 '내용 시작' 위치: cl100k 토큰은 앞 공백을 포함하므로(" The"), 공백을 건너뛴 위치로 문장·환각 구간에 배정한다.
    # (v1에서는 공백 위치로 배정해 문장 경계 토큰이 앞 문장에 붙는 오류가 발생 — 발생 위치의 18%)
    cs = [a + (len(text[a:b]) - len(text[a:b].lstrip())) for a, b in zip(ts, te)]
    cs = [c if c < b else a for c, a, b in zip(cs, ts, te)]          # 공백뿐인 토큰
    spans = [(l['start'], l['end']) for l in (r.get('labels') or [])]
    bad = [j for j in range(len(ts)) if any(min(te[j], b) > max(cs[j], a) for a, b in spans)]
    sents = []
    for sp in seg.segment(text):
        if not sp.sent.strip(): continue
        a, b = sp.start, sp.end
        tok = [j for j in range(len(ts)) if a <= cs[j] < b]
        if not tok: continue
        y = int(any(min(b, e) > max(a, s) for s, e in spans))
        sents.append({'a': a, 'b': b, 't0': tok[0], 't1': tok[-1], 'y': y})
    sent_of = {t: sn['t0'] for sn in sents for t in range(sn['t0'], sn['t1'] + 1)}
    return {'id': str(r['id']), 'task_type': r['task_type'], 'gen_model': r['model'],
            'ref_key': r['source_id'], 'text': text, 'ts': ts, 'te': te, 'n_tok': len(ts),
            'onset': bad[0] if bad else None, 'sents': sents, 'sent_of': sent_of,
            'ends': {sn['t1'] for sn in sents}}

REFS = {}
def to_items(rs):
    out = []
    for r in rs:
        REFS.setdefault(r['source_id'], build_reference(sources[r['source_id']]))
        out.append(prep(r))
    return out

dev_items, val_items = to_items(dev), to_items(val)
for name, it in [('dev', dev_items), ('검증', val_items)]:
    n_h = sum(x['onset'] is not None for x in it)
    print(f'{name}: 응답 {len(it)} (환각 {n_h}, 없음 {len(it)-n_h}) | 토큰 {sum(x["n_tok"] for x in it):,} | '
          f'환각 없는 응답 토큰 {sum(x["n_tok"] for x in it if x["onset"] is None):,}')

In [ ]:
TRAIN_ITEMS = to_items(train)
ITEMS = {f'tr-{x["id"]}': x for x in TRAIN_ITEMS}
LABELS = {k: {int(i): v for i, v in d.items()} for k, d in json.load(open(os.path.join(R4B_DIR, 'labels_train.json'))).items()}
print(f'학습 재판정 {len(LABELS):,}/{len(ITEMS):,}')

In [ ]:
def content_starts(x):
    t = x['text']
    cs = [a + (len(t[a:b]) - len(t[a:b].lstrip())) for a, b in zip(x['ts'], x['te'])]
    return [c if c < b else a for c, a, b in zip(cs, x['ts'], x['te'])]

def locate(x, sn, phrase, cs):
    seg = x['text'][sn['a']:sn['b']]
    ph = (phrase or '').strip()
    k = seg.find(ph) if ph else -1
    if k < 0 and ph: k = seg.lower().find(ph.lower())
    if k < 0: return sn['t0']
    a, b = sn['a'] + k, sn['a'] + k + len(ph)
    for t in range(sn['t0'], sn['t1'] + 1):
        if min(x['te'][t], b) > max(cs[t], a): return t
    return sn['t0']

ADD_ONSET, flip, earlier = {}, 0, 0
for cid, x in ITEMS.items():
    labs = LABELS.get(cid)
    if not labs: continue
    cs, add = None, []
    for i, sn in enumerate(x['sents']):
        if sn['y'] == 0 and labs[i + 1]['label'] == 'H':
            cs = cs or content_starts(x)
            t = locate(x, sn, labs[i + 1]['phrase'], cs); ADD_ONSET[(x['id'], i)] = t; add.append(t)
    if add:
        if x['onset'] is None: flip += 1
        elif min(add) < x['onset']: earlier += 1
n_clean = sum(x['onset'] is None for x in TRAIN_ITEMS); n_hall = len(TRAIN_ITEMS) - n_clean
print(f'추가된 환각 문장 {len(ADD_ONSET):,} | 멀쩡 → 환각 {flip}/{n_clean} ({flip / max(n_clean, 1):.0%}) | 더 이른 환각 {earlier}/{n_hall} ({earlier / max(n_hall, 1):.0%})')

## 3. 고쳐 쓸 문장 고르기

In [ ]:
# 고쳐 쓸 문장 고르기: 보정 학습 정답(R4b Sonnet) 기준 멀쩡한 문장(원래 정답 y=0, 추가 환각 아님, 판정 S), 8~60토큰
N_PER_TASK, MAX_PER_RESP, MIN_TOK, MAX_TOK = 500, 2, 8, 60
HOLD = {x['ref_key'] for k in ['calL', 'devL'] for x in R0P['items'][k]}
assert not HOLD & {x['ref_key'] for x in TRAIN_ITEMS}, '학습 응답에 cal-L·dev-L 근거가 섞임'

def sent_text(x, sn): return x['text'][sn['a']:sn['b']].strip()

def candidates(x):
    labs = LABELS.get(f'tr-{x["id"]}')
    if not labs: return []
    return [i for i, sn in enumerate(x['sents'])
            if sn['y'] == 0 and (x['id'], i) not in ADD_ONSET and labs[i + 1]['label'] == 'S'
            and MIN_TOK <= sn['t1'] - sn['t0'] + 1 <= MAX_TOK]

rs = random.Random(2028)
PICK = {}
for tt in TASKS:
    pool = [x for x in TRAIN_ITEMS if x['task_type'] == tt and candidates(x)]
    for x in rs.sample(pool, min(N_PER_TASK, len(pool))):
        c = candidates(x)
        PICK[x['id']] = sorted(rs.sample(c, min(MAX_PER_RESP, len(c))))
BY_ID = {x['id']: x for x in TRAIN_ITEMS}
print(f'고쳐 쓸 응답 {len(PICK)}개, 문장 {sum(len(v) for v in PICK.values())}개 |',
      {t: sum(BY_ID[i]['task_type'] == t for i in PICK) for t in TASKS})

## 4. Claude API (M1과 같은 배치 코드)

In [ ]:
import anthropic
def get_key(name):
    try:
        from google.colab import userdata
        v = userdata.get(name)
        if v: return v
    except Exception:
        pass
    from getpass import getpass
    return getpass(f'{name}: ')
A = anthropic.Anthropic(api_key=get_key('ANTHROPIC_API_KEY'), max_retries=3)
PRICE = {'claude-haiku-5-5': (0.05, 0.25), 'claude-sonnet-5-5': (1.0, 5.0), 'claude-opus-5-5': (2.0, 10.0)}   # 배치 $/백만 토큰 (2026-11)

def text_of(m): return ''.join(b.text for b in m.content if getattr(b, 'type', '') == 'text')
def rec(m): return {'text': text_of(m), 'stop': m.stop_reason, 'model': m.model, 'in': m.usage.input_tokens, 'out': m.usage.output_tokens}
def cost(res): return sum(r['in'] / 1e6 * PRICE.get(r['model'], (2, 10))[0] + r['out'] / 1e6 * PRICE.get(r['model'], (2, 10))[1] for r in res.values())

def run_batch(reqs, path, chunk=400):
    # reqs: cid -> 요청. 결과: cid -> rec. 다시 실행하면 제출한 묶음·받은 결과는 건너뛴다
    res = load_json(path) or {}
    bpath = path.replace('.json', '_batches.json')
    B = load_json(bpath) or {'chunks': []}
    sent = {c for ch in B['chunks'] for c in ch['cids']}
    todo = [c for c in reqs if c not in sent and c not in res]
    for k in range(0, len(todo), chunk):
        part = todo[k:k + chunk]
        b = A.messages.batches.create(requests=[{'custom_id': c, 'params': reqs[c]} for c in part])
        B['chunks'].append({'id': b.id, 'cids': part, 'done': False}); save_json(B, bpath)
        print(f'  제출 {b.id} ({len(part)}건)', flush=True)
    for ch in B['chunks']:
        if ch.get('done'): continue
        while True:
            b = A.messages.batches.retrieve(ch['id']); rc = b.request_counts
            print(f'  {time.strftime("%H:%M:%S")} {ch["id"][-8:]} {b.processing_status} | 처리 중 {rc.processing}, 성공 {rc.succeeded}, 오류 {rc.errored}', flush=True)
            if b.processing_status == 'ended': break
            time.sleep(60)
        for r in A.messages.batches.results(ch['id']):
            if r.result.type == 'succeeded': res[r.custom_id] = rec(r.result.message)
        ch['done'] = True; save_json(B, bpath); save_json(res, path)
    miss = [c for c in reqs if c not in res]
    for c in miss[:100]:                                   # 배치 실패분은 바로 요청 1회
        try: res[c] = rec(A.messages.create(**reqs[c]))
        except Exception as e: print(f'  {c}: {type(e).__name__} {str(e)[:150]}'); break
    save_json(res, path)
    print(f'  결과 {len(res)}/{len(reqs)} | 사용량 약 ${cost({c: res[c] for c in reqs if c in res}):.2f}')
    return res

## 5. ① 모순 · ② 바꿔쓰기 생성

In [ ]:
SYSTEM = '''You are an expert annotator for RAGTruth, a benchmark of hallucinations in retrieval-augmented generation.
You receive a REFERENCE that a language model was given, and the model's RESPONSE split into numbered sentences.
Label EVERY sentence against the REFERENCE only (not world knowledge).

A sentence contains a hallucination if any part of it is:
- Evident conflict: clearly contradicts the reference (wrong fact, number, name, date, negation).
- Subtle conflict: deviates from the reference in a way that changes meaning (altered term, scope, time, attribution).
- Evident baseless info: introduces information that is not in the reference, even if it may be true in the real world.
- Subtle baseless info: adds unsupported inference, opinion, or embellishment (e.g. "popular", "cozy", "family-friendly") that the reference does not support.

Not hallucinations: faithful paraphrase; reasonable summarization; statements that the reference lacks the information or declining to answer;
meta phrases such as "Based on the passages,". For Data2txt (business data + user reviews), claims supported by the structured data or by what reviewers say are supported.

Labels: "S" = no hallucination. "H" = contains a hallucination. "A" = ambiguous, careful annotators could reasonably disagree.
For every H or A sentence also give: "type" (one of: evident_conflict, subtle_conflict, evident_baseless, subtle_baseless),
"phrase" (the shortest span copied EXACTLY, character for character, from that sentence that is hallucinated), and "reason" (at most 20 words).

Return ONLY a JSON object, no other text:
{"sentences": [{"i": 1, "label": "S"}, {"i": 2, "label": "H", "type": "evident_baseless", "phrase": "...", "reason": "..."}]}
with exactly one entry per numbered sentence, in order.'''

TASK_DESC = {'QA': 'QA — answer the question using the passages',
             'Summary': 'Summary — summarize the news article',
             'Data2txt': 'Data2txt — write an overview of a business from structured data and user reviews'}

SYN_MODEL = 'claude-sonnet-5-5'
SYN_SYSTEM = '''You create training data for a hallucination detector used in retrieval-augmented generation.
You receive a REFERENCE (the only source the model was given), a model RESPONSE split into numbered sentences, and TARGET sentence numbers.
Every target sentence is faithful to the reference. For EACH target sentence write two new versions of it:

1. "contradiction": a MINIMAL EDIT of the sentence that now DIRECTLY CONFLICTS with the reference.
   - Keep most of the original words and the sentence structure. Change one relation or detail so a careful reader who checks the reference finds it wrong.
   - It must contradict the reference, not merely add unsupported information. It must stay fluent and plausible.
   - Preferred kinds by task:
     Data2txt: attribution (which reviewer / how many reviewers said it, e.g. "another reviewer" when it was the same one, "several" when only one),
               flipped review detail (who did what, what was praised or criticized), mismatch with the business fields (hours, attributes, category).
     Summary: swapped roles or entities (who did what to whom, which person held which title), changed cause/effect, time or order, quantity.
     QA: changed step, condition or detail in the procedure or fact; negation; wrong item from the passages.
   - "changed": the shortest span copied EXACTLY from your contradiction sentence that carries the conflict.
   - "kind": one short label for what you changed (e.g. attribution, role_swap, detail_flip, negation, number, time, field_mismatch).
2. "paraphrase": a faithful rewording of the ORIGINAL sentence with the same meaning, fully supported by the reference.
   Reword noticeably (different wording or order) but add or drop no information.

Return ONLY a JSON object, no other text:
{"items": [{"i": 3, "contradiction": "...", "changed": "...", "kind": "...", "paraphrase": "..."}]}
with exactly one entry per target sentence.'''

def numbered(sents):
    return '\n'.join(f'[{k + 1}] {s}' for k, s in enumerate(sents))

def syn_msg(x, targets):
    sents = [sent_text(x, sn) for sn in x['sents']]
    return (f"TASK: {TASK_DESC[x['task_type']]}\n\nREFERENCE:\n<<<\n{REFS[x['ref_key']]}\n>>>\n\n"
            f"RESPONSE ({len(sents)} numbered sentences):\n{numbered(sents)}\n\nTARGET sentences: {', '.join(str(i + 1) for i in targets)}")

def syn_params(x, targets):
    return {'model': SYN_MODEL, 'max_tokens': 3000, 'system': SYN_SYSTEM,
            'messages': [{'role': 'user', 'content': syn_msg(x, targets)}]}

def words(s): return [w.strip('.,;:!?"\'()').lower() for w in s.split() if w.strip('.,;:!?"\'()')]

def overlap(a, b):
    # 고쳐 쓴 문장이 원래 단어를 얼마나 유지했나 (원래 단어 중 남은 비율)
    wa, wb = words(a), set(words(b))
    return sum(w in wb for w in wa) / max(len(wa), 1)

def parse_syn(text, x, targets):
    # 후보 목록: (문장 번호 0부터, 종류 contra/para, 새 문장, 바뀐 구절, 바꾼 종류). 형식이 틀린 항목은 버린다
    try:
        d = json.loads(text[text.find('{'): text.rfind('}') + 1])
    except Exception:
        return [], 'json'
    out = []
    for e in d.get('items', []):
        try: i = int(e.get('i', 0)) - 1
        except Exception: continue
        if i not in targets: continue
        orig = sent_text(x, x['sents'][i])
        c, ch, p = str(e.get('contradiction', '')).strip(), str(e.get('changed', '')).strip(), str(e.get('paraphrase', '')).strip()
        if c and ch and c != orig and (ch in c or ch.lower() in c.lower()) and overlap(orig, c) >= 0.5:
            out.append((i, 'contra', c, ch, str(e.get('kind', ''))[:30]))
        if p and p != orig:
            out.append((i, 'para', p, '', ''))
    return out, 'ok'

# 형식 확인 1건 (수 센트)
_x = BY_ID[next(iter(PICK))]
_g = parse_syn(text_of(A.messages.create(**syn_params(_x, PICK[_x['id']]))), _x, PICK[_x['id']])
for i, k, s, ch, kd in _g[0]: print(f"  [{i + 1}] {k:6} {kd:14} {s[:150]!r} | 바뀐 구절 {ch!r}")
print(f"예상 비용: 생성 약 ${len(PICK) * (2500 * 1.0 + 700 * 5.0) / 1e6:.1f} + 검증 약 ${len(PICK) * 4 * (2600 * 1.0 + 80 * 5.0) / 1e6:.1f} (토큰 추정)")

GEN = run_batch({f'syn-{xid}': syn_params(BY_ID[xid], t) for xid, t in PICK.items()}, os.path.join(K1_DIR, 'syn_gen.json'))
CANDS, bad = {}, 0
for xid, t in PICK.items():
    g = GEN.get(f'syn-{xid}')
    if g is None: continue
    out, st = parse_syn(g['text'], BY_ID[xid], t)
    bad += st != 'ok'
    for i, kind, s, ch, kd in out:
        x = BY_ID[xid]
        CANDS[f'v-{xid}-{i}-{kind}'] = {'rid': xid, 'ref_key': x['ref_key'], 'task': x['task_type'], 'i': i, 'kind': kind,
                                        'orig': sent_text(x, x['sents'][i]), 'sentence': s, 'phrase': ch, 'gen_kind': kd}
print(f"후보: 모순 {sum(c['kind'] == 'contra' for c in CANDS.values())}, 바꿔쓰기 {sum(c['kind'] == 'para' for c in CANDS.values())} | JSON 파싱 실패 {bad}")

## 6. 따로 판정해 검증 (R4 지시문, 대상 문장만)

In [ ]:
VERIFY_MODEL = 'claude-sonnet-5-5'

def verify_msg(x, i, new):
    sents = [sent_text(x, sn) for sn in x['sents']]
    sents[i] = new
    return (f"TASK: {TASK_DESC[x['task_type']]}\n\nREFERENCE:\n<<<\n{REFS[x['ref_key']]}\n>>>\n\n"
            f"RESPONSE ({len(sents)} numbered sentences):\n{numbered(sents)}\n\n"
            f"This time label ONLY sentence [{i + 1}]. Return ONLY "
            '{"sentences": [{"i": ' + str(i + 1) + ', "label": "S"}]} with that single entry (add type, phrase and reason if H or A).')

def verify_params(x, i, new):
    # R4와 같은 정답 지시문(SYSTEM), 대상 문장 하나만
    return {'model': VERIFY_MODEL, 'max_tokens': 400, 'system': SYSTEM,
            'messages': [{'role': 'user', 'content': verify_msg(x, i, new)}]}

def parse_verify(text, i):
    try:
        d = json.loads(text[text.find('{'): text.rfind('}') + 1])
        for e in d.get('sentences', []):
            if int(e.get('i', 0)) == i + 1:
                return str(e.get('label', '')).strip().upper()[:1] or None
    except Exception:
        pass
    return None

def keep(kind, label):
    return (kind == 'contra' and label == 'H') or (kind == 'para' and label == 'S')

VRES = run_batch({vid: verify_params(BY_ID[c['rid']], c['i'], c['sentence']) for vid, c in CANDS.items()},
                 os.path.join(K1_DIR, 'syn_verify.json'))

## 7. 저장 · K1-0(개수) 확인 · 점검 표본

In [ ]:
N_MAX = 1500
def build_synth(cands, vres):
    # cands: vid -> 후보 dict, vres: vid -> 판정 결과. 통과한 ①·② 중 같은 수(최대 각 N_MAX)
    ok = {'contra': [], 'para': []}
    for vid, c in cands.items():
        if vid not in vres: continue
        lab = parse_verify(vres[vid]['text'], c['i'])
        c = dict(c, verify=lab)
        if keep(c['kind'], lab): ok[c['kind']].append(c)
    n = min(len(ok['contra']), len(ok['para']), N_MAX)
    r = random.Random(2029)
    out = r.sample(ok['contra'], n) + r.sample(ok['para'], n)
    return out, {k: len(v) for k, v in ok.items()}, n

def check_sample(synth, k=40):
    # K1-0 가린 점검용: 통과한 모순 40개 (근거, 원래 문장, 고친 문장, 바뀐 구절)
    con = [c for c in synth if c['kind'] == 'contra']
    sel = random.Random(2030).sample(con, min(k, len(con)))
    return [{'qid': f'q{n + 1:02d}', 'task': c['task'], 'reference': REFS[c['ref_key']], 'original': c['orig'],
             'rewritten': c['sentence'], 'changed': c['phrase']} for n, c in enumerate(sel)]

SYN, PASSED, N_EACH = build_synth(CANDS, VRES)
by = pd.DataFrame([dict(c, verify=parse_verify(VRES[v]['text'], c['i']) if v in VRES else None) for v, c in CANDS.items()])
print('검증 결과 (종류 × 판정)'); print(pd.crosstab([by.task, by.kind], by.verify.fillna('없음')))
print('모순 종류(통과분)', pd.Series([c['gen_kind'] for c in SYN if c['kind'] == 'contra']).value_counts().head(12).to_dict())
K1_0_count = PASSED['contra'] >= 300
print(f"=== K1-0(개수): 통과한 모순 {PASSED['contra']} (≥ 300) → {'충족' if K1_0_count else '미달 → 보류'} | 학습에 쓸 수 각 {N_EACH} ===")
save_json(SYN, os.path.join(K1_DIR, 'synth_train.json'))
LOCAL = '/content/k1a_local'; os.makedirs(LOCAL, exist_ok=True)
CHECK = check_sample(SYN)
json.dump(CHECK, open(os.path.join(LOCAL, 'k1_check.json'), 'w'), ensure_ascii=False, indent=1)
usage = {f: round(cost(load_json(os.path.join(K1_DIR, f))), 2) for f in ['syn_gen.json', 'syn_verify.json']}
summ = {'n_pick_resp': len(PICK), 'n_pick_sent': sum(len(v) for v in PICK.values()), 'cands': by.kind.value_counts().to_dict(),
        'passed': PASSED, 'n_each': N_EACH, 'K1_0_count': bool(K1_0_count), 'cost_usd': usage,
        'by_task': {t: {k: int(((by.task == t) & (by.kind == k) & by.apply(lambda r: keep(r['kind'], r['verify']), axis=1)).sum()) for k in ['contra', 'para']} for t in TASKS},
        'gen_kinds': pd.Series([c['gen_kind'] for c in SYN if c['kind'] == 'contra']).value_counts().to_dict()}
json.dump(summ, open(os.path.join(LOCAL, 'k1a_summary.json'), 'w'), ensure_ascii=False, indent=1)
print('API 사용량(추정 $)', usage)
import shutil
zp = shutil.make_archive('/content/k1a_results', 'zip', LOCAL)
from google.colab import files
files.download(zp)